
## Notobook  for Data Imputation 



In [1]:
import zipfile
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

In [2]:
# Define a function to create sequences
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:i+seq_length]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

class LSTMImputer(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super(LSTMImputer, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out
    

In [ ]:
# load data

zip_file = ['242','243','245','246','247','252','253','254','255','266']


for csv in zip_file:
    # Load the CSV file into a DataFrame
    data = pd.read_csv('sigId-' + csv + '.csv', usecols=['Voltage.Va', 'Voltage.Vb', 'Voltage.Vc'])
    data_origin = data.copy() # complete data

    # Define the proportion of data to be replaced with NaN
    nan_proportion = 0.01  # 1% of the data

    # Calculate the total number of elements
    total_elements = data.shape[0] * 3  # Number of rows * 3 columns

    # Determine the number of elements to replace with NaN
    num_nan = int(total_elements * nan_proportion)

    # Randomly select the indices to replace with NaN
    nan_indices = (
        np.random.randint(0, data.shape[0], num_nan),  # Random row indices
        np.random.randint(0, 3, num_nan)  # Random column indices
    )

    # Replace selected elements in the first two columns with NaN
    data.iloc[nan_indices] = np.nan

    df = pd.DataFrame(data)

    # Forward fill to handle NaN temporarily
    df_filled = df.fillna(method='ffill').fillna(method='bfill')

    # Normalize the data
    scaler = RobustScaler()
    data_scaled = scaler.fit_transform(df_filled)


    seq_length = 2  # Adjust this based on the nature of your data
    X, y = create_sequences(data_scaled, seq_length)


    # Convert to PyTorch tensors
    X_train = torch.tensor(X, dtype=torch.float32)
    y_train = torch.tensor(y, dtype=torch.float32)

    # Initialize and train the model
    input_size = X_train.shape[2]
    hidden_size = 80
    num_layers = 2
    model = LSTMImputer(input_size, hidden_size, num_layers)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    num_epochs = 500
    for epoch in range(num_epochs):
        model.train()
        outputs = model(X_train)
        optimizer.zero_grad()
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}')

    # Prepare the full sequence for prediction
    X_full, _ = create_sequences(data_scaled, seq_length)
    X_full = torch.tensor(X_full, dtype=torch.float32)

    # Make predictions
    model.eval()
    with torch.no_grad():
        predictions = model(X_full).numpy()

    # Combine the predicted values with the original data
    imputed_data = np.copy(data_scaled)
    imputed_data[seq_length:, :] = predictions  # Replace from seq_length onwards

    # Reverse the scaling
    imputed_data = scaler.inverse_transform(imputed_data)

    # Replace only NaN values in the original data
    df_final = df.copy()
    for i in range(df.shape[1]):
        mask = df.iloc[:, i].isna()
        df_final.loc[mask, df.columns[i]] = imputed_data[mask, i]

    print(df_final)

    for col in data_origin.columns:
        rmse = np.sqrt(mean_squared_error(data_origin[col].dropna(), df_final[col].dropna()))
        data_range = data_origin[col].max() - data_origin[col].min()
        rmse_range_ratio = rmse / data_range * 100

        print(f'Column: {col}')
        print(f'RMSE: {rmse_range_ratio:.4f}%')
        print('-' * 40)


    # Plot the DataFrame
    # Create a 3x2 grid of subplots
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 10))

    # Plotting on each subplot
    axes[0, 0].plot(df['Voltage.Va'])  # Top-left
    axes[0, 0].set_title('Va')

    axes[0, 1].plot(df_final['Voltage.Va'])  # Top-right
    axes[0, 1].set_title('Imputed Va')

    axes[1, 0].plot(df['Voltage.Vb'])  # Middle-left
    axes[1, 0].set_title('Vb')

    axes[1, 1].plot(df_final['Voltage.Vb'])  # Middle-right
    axes[1, 1].set_title('Imputed Vb')

    axes[2, 0].plot(df['Voltage.Vc'])  # Bottom-left
    axes[2, 0].set_title('Vc')

    axes[2, 1].plot(df_final['Voltage.Vc'])  # Bottom-right
    axes[2, 1].set_title('Imputed Vc')

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Save the figure
    plt.savefig(csv + '.png', format='png', dpi=300, bbox_inches='tight')

    plt.show()